In [ ]:
import pandas as pd
import sqlite3

In [ ]:
AUTORES = ['Eduardo Ziegler Berti', 'Mateus Soares Lopes de Oliveira']

In [ ]:
def carrega_tabela(database, tabela):
    '''
    Carrega uma tabela de um banco de dados 
    SQLite em um DataFrame do Pandas.
    Parâmetros:
    database: str - Caminho para o arquivo do banco de dados SQLite.
    tabela: str - Nome da tabela a ser carregada.
    '''
    with sqlite3.connect(database) as conn:
        query = f"SELECT * FROM {tabela}"
        df = pd.read_sql_query(query, conn)
    return df


In [ ]:
def lista_tabelas(db_filename):
    '''
    Lista todas as tabelas em um banco de dados SQLite.
    Parâmetros:
    db_filename: str - Caminho para o arquivo do banco de dados SQLite.
    Retorna:
    DataFrame com os nomes das tabelas e suas contagens de linhas.
    '''
    conn = sqlite3.connect(db_filename)
    with sqlite3.connect(db_filename) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = cursor.fetchall()

        table_row_counts = []
        for table in tables:
            table_name = table[0]
            query = f"SELECT COUNT(*) FROM {table_name};"
            cursor.execute(query)
            row_count = cursor.fetchone()[0]
            table_row_counts.append({"Table": table_name, "Row_Count": row_count})

        return pd.DataFrame(table_row_counts)

In [ ]:
def questao_1(database):
    sessao = carrega_tabela(database, 'sessao')
    resultado = int(sessao.publico.sum())
    return resultado

questao_1('bilheteria.db')

110153083

In [ ]:
def questao_2(database):
    sessao = carrega_tabela(database, 'sessao')[['filme_id', 'publico']]
    filme = carrega_tabela(database, 'filme')

    uniao = sessao.merge(filme[['id','titulo_original', 'pais_origem']], left_on='filme_id', right_on='id')
    tabela = uniao.groupby(['titulo_original', 'pais_origem'])['publico'].sum()

    tabela_ordenada = tabela.reset_index().sort_values('publico', ascending=False).drop_duplicates(subset=['pais_origem'])
    resultado = tabela_ordenada.reset_index()[['titulo_original', 'pais_origem']].rename(columns={
        'titulo_original':'TíÍTULO',
        'pais_origem':'PAÍS DE ORIGEM'})
    return resultado

questao_2('bilheteria.db')

,TíÍTULO,PAÍS DE ORIGEM
0,BARBIE,ESTADOS UNIDOS
1,PAW PATROL: THE MIGHTY MOVIE,CANADÁ
2,NAPOLEON,INGLATERRA
3,TALK TO ME,AUSTRÁLIA
4,NOSSO SONHO,BRASIL
5,MUMMIES,ESPANHA
6,KNIGHTS OF THE ZODIAC,JAPÃO
7,OPERATION FORTUNE: RUSE DE GUERRE,CHINA
8,THE THREE MUSKETEERS: D’ARTAGNAN,FRANÇA
9,CATS IN THE MUSEUM,BELARUS (BIELORUSSIA)


In [ ]:
def questao_3(database):
    complexo = carrega_tabela(database, 'complexo')[['municipio', 'id']]
    sala = carrega_tabela(database, 'sala')[['id', 'from_complexo']].rename(columns={'id': 'sala_id'})
    # o id se repetia, sendo que o id do complexo é diferente do id da sala, mas o id da sala é igual a sala_id
    sessao = carrega_tabela(database, 'sessao')[['sala_id', 'publico']]
    
    uniao_complexo_sala = complexo.merge(sala, left_on='id', right_on='from_complexo')
    uniao_tabela = uniao_complexo_sala.merge(sessao, left_on='sala_id', right_on='sala_id')[['municipio', 'publico']]
    tabela = uniao_tabela.groupby('municipio')['publico'].sum()
    tabela_ordenada = tabela.reset_index().sort_values('publico', ascending=False).head(100).reset_index(drop=True)
    resultado = tabela_ordenada.rename(columns={
        'municipio': 'MUNICIPIO',
        'publico': 'PÚBLICO'
    })

    return resultado

questao_3('bilheteria.db')

,MUNICIPIO,PÚBLICO
0,SÃO PAULO,13310470
1,RIO DE JANEIRO,8496306
2,BRASÍLIA,3272437
3,BELO HORIZONTE,3009414
4,CURITIBA,2965114
...,...,...
95,ARAÇATUBA,247992
96,VITÓRIA DA CONQUISTA,246490
97,PONTA GROSSA,241142
98,LAURO DE FREITAS,240784


In [ ]:
def questao_4(database):
    complexo = carrega_tabela(database, 'complexo')[['municipio', 'id']]
    sala = carrega_tabela(database, 'sala')[['id', 'from_complexo']].rename(columns={'id': 'sala_id'})
    filme = carrega_tabela(database, 'filme')[['id', 'titulo_original']].rename(columns={'id': 'filme_id'})
    sessao = carrega_tabela(database, 'sessao')[['sala_id', 'filme_id', 'publico']]
    uniao_complexo_sala = complexo.merge(sala, left_on='id', right_on='from_complexo')
    uniao = uniao_complexo_sala.merge(sessao, on='sala_id')[['municipio', 'filme_id', 'publico']]
    uniao = uniao.merge(filme, on='filme_id')[['municipio', 'titulo_original', 'publico']]
    tabela = uniao.groupby(['municipio', 'titulo_original'])['publico'].sum().reset_index()
    idx = tabela.groupby('municipio')['publico'].idxmax()
    resultado = tabela.loc[idx].rename(columns={
        'municipio': 'CIDADE',
        'titulo_original': 'FILME',
        'publico': 'BILHETERIA'
    })
    resultado = resultado.sort_values('BILHETERIA', ascending=False).reset_index(drop=True)
    return resultado
questao_4('bilheteria.db')

,CIDADE,FILME,BILHETERIA
0,SÃO PAULO,BARBIE,1201206
1,RIO DE JANEIRO,BARBIE,719539
2,BRASÍLIA,BARBIE,289033
3,BELO HORIZONTE,BARBIE,283911
4,CURITIBA,BARBIE,269314
...,...,...,...
340,CAMPOS NOVOS,FAST X,654
341,MOGI-GUACU,THE EXPENDABLES 4,601
342,CÓRREGO FUNDO,FRANCO NO TREM DO MEDO,53
343,PAINS,FRANCO NO TREM DO MEDO,51


In [ ]:
def questao_5(database):
    complexo = carrega_tabela(database, 'complexo')[['municipio', 'id']]
    sala = carrega_tabela(database, 'sala')[['id', 'from_complexo']].rename(columns={'id': 'sala_id'})
    filme = carrega_tabela(database, 'filme')[['id', 'pais_origem']].rename(columns={'id': 'filme_id'})
    sessao = carrega_tabela(database, 'sessao')[['sala_id', 'filme_id', 'publico']]
    uniao = complexo.merge(sala, left_on='id', right_on='from_complexo')
    uniao = uniao.merge(sessao, on='sala_id')
    uniao = uniao.merge(filme, on='filme_id')[['municipio', 'pais_origem', 'publico']]
    br = uniao[uniao['pais_origem'] == 'BRASIL'].groupby('municipio')['publico'].sum().rename('BILHETERIA_BR')
    estrangeiro = uniao[uniao['pais_origem'] != 'BRASIL'].groupby('municipio')['publico'].sum().rename('BILHETERIA_ESTRANGEIRA')
    resultado = pd.concat([br, estrangeiro], axis="columns").fillna(0).reset_index()
    resultado = resultado.rename(columns={'municipio': 'CIDADE'})
    resultado[['BILHETERIA_BR', 'BILHETERIA_ESTRANGEIRA']] = resultado[['BILHETERIA_BR', 'BILHETERIA_ESTRANGEIRA']].astype(int)
    resultado = resultado.sort_values(by='BILHETERIA_BR', ascending=False)
    return resultado
questao_5('bilheteria.db')

,CIDADE,BILHETERIA_BR,BILHETERIA_ESTRANGEIRA
257,RIO DE JANEIRO,546789,7949517
303,SÃO PAULO,403251,12907219
262,SALVADOR,166624,2304396
39,BELO HORIZONTE,98746,2910668
250,RECIFE,84134,2248439
...,...,...,...
340,CANOINHAS,0,19082
341,ITAPEVA,0,19472
342,JABOTICABAL,0,16460
343,LINS,0,18654
